# RL Teaching — Day 2
**CartPole · FrozenLake Slippery Comparison**

> 📌 **Before you start:** Go to **File → Save a copy in Drive** to make your own editable copy.

Focus on **what changes when you adjust the 🔧 parameters** and re-run the cell.

In [ ]:
# ── Setup ────────────────────────────────────────────────────
!pip install gymnasium matplotlib numpy --quiet
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
print('✅ Setup complete!')

---
## Part 1 · CartPole — Continuous State Space

CartPole: keep a pole balanced on a moving cart.
The state has **4 continuous values** — not grid squares like FrozenLake.

Q-learning needs *discrete* states, so we divide each dimension into **bins**.
Rein Room does this automatically; here you can see exactly what happens.

```
State dimension    | Range          | Bins
Cart position      | -2.4  to  2.4  | 10
Cart velocity      | -3.0  to  3.0  | 10
Pole angle         | -0.25 to  0.25 | 10
Pole ang. velocity | -3.0  to  3.0  | 10
Total possible states: 10 x 10 x 10 x 10 = 10,000
```

### 🔧 Your task
Watch the reward curve. When does improvement start?
Does it ever drop suddenly after improving? Why might that happen?

In [ ]:
# ════════════════════════════════════════════
alpha      = 0.5   # 🔧 learning rate    — try: 0.2 / 0.5 / 0.8
gamma      = 0.99  # 🔧 discount factor  — try: 0.9 / 0.99
epsilon    = 0.3   # 🔧 exploration rate — try: 0.1 / 0.3 / 0.5
n_episodes = 500
# ════════════════════════════════════════════

env    = gym.make('CartPole-v1')
n_bins = 10
bins   = [
    np.linspace(-2.4,  2.4,  n_bins),
    np.linspace(-3.0,  3.0,  n_bins),
    np.linspace(-0.25, 0.25, n_bins),
    np.linspace(-3.0,  3.0,  n_bins),
]

def discretize(obs):
    return tuple(min(np.digitize(obs[i], bins[i]), n_bins-1) for i in range(4))

Q      = {}
get_q  = lambda s, a: Q.get((s, a), 0.0)
best_a = lambda s: max(range(2), key=lambda a: get_q(s, a))
episode_rewards, episode_lengths = [], []

for ep in range(n_episodes):
    obs, _  = env.reset()
    state   = discretize(obs)
    total, steps, done = 0, 0, False
    while not done:
        action = env.action_space.sample() if np.random.random() < epsilon else best_a(state)
        nobs, r, ter, tru, _ = env.step(action)
        done   = ter or tru
        ns     = discretize(nobs)
        old_q  = get_q(state, action)
        nmax   = max(get_q(ns, a) for a in range(2))
        Q[(state, action)] = old_q + alpha * (r + gamma * nmax - old_q)
        state, total, steps = ns, total + r, steps + 1
    episode_rewards.append(total)
    episode_lengths.append(steps)

window = 30
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(np.convolve(episode_rewards, np.ones(window)/window, mode='valid'), lw=2)
ax1.axhline(195, color='red', linestyle='--', label='Solved threshold (195)')
ax1.set(title=f'CartPole Reward | alpha={alpha} gamma={gamma} epsilon={epsilon}',
        xlabel='Episode', ylabel='Total Reward (max 200)')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(np.convolve(episode_lengths, np.ones(window)/window, mode='valid'), color='orange', lw=2)
ax2.set(title='Episode Length  (longer = better balance)',
        xlabel='Episode', ylabel='Steps')
ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Q-table entries used : {len(Q):,} / {10**4*2:,} possible')
print(f'Avg reward (last 50) : {np.mean(episode_rewards[-50:]):.1f} / 200')

---
## Part 2 · FrozenLake — Slippery vs. Non-Slippery

Previously: `is_slippery=False` — agent moves exactly where intended.
Now: `is_slippery=True` — agent sometimes slides sideways.

### 🔧 Your task
Compare the two curves:
- Which converges faster?
- Which is more stable (less bouncing)?
- What does this tell you about learning in unpredictable environments?

In [ ]:
alpha, gamma, epsilon, n_episodes = 0.8, 0.95, 0.1, 3000
window = 100

def run_frozenlake(slippery):
    env = gym.make('FrozenLake-v1', is_slippery=slippery)
    Q   = np.zeros([env.observation_space.n, env.action_space.n])
    rewards = []
    for _ in range(n_episodes):
        state, _ = env.reset()
        total, done = 0, False
        while not done:
            action = env.action_space.sample() if np.random.random() < epsilon else np.argmax(Q[state])
            ns, r, ter, tru, _ = env.step(action)
            done = ter or tru
            Q[state, action] += alpha * (r + gamma * np.max(Q[ns]) - Q[state, action])
            state, total = ns, total + r
        rewards.append(total)
    return rewards

print('Training both environments... (~15 seconds)')
r_det  = run_frozenlake(slippery=False)
r_slip = run_frozenlake(slippery=True)

plt.figure(figsize=(12, 4))
for rewards, label, color in [
    (r_det,  'Non-slippery (deterministic)', 'steelblue'),
    (r_slip, 'Slippery (stochastic)',         'coral'),
]:
    s = np.convolve(rewards, np.ones(window)/window, mode='valid')
    plt.plot(s, label=label, color=color, lw=2)
plt.title('FrozenLake: Deterministic vs. Stochastic')
plt.xlabel('Episode'); plt.ylabel('Success Rate (smoothed)')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f'Non-slippery success (last 500): {np.mean(r_det[-500:]):.1%}')
print(f'Slippery     success (last 500): {np.mean(r_slip[-500:]):.1%}')